# 📱 GUARDIAN — 폰 실시간 ADAS HUD (v6.5) Colab 헬퍼

**프로젝트:** AIFFEL ML 과정 — 모터사이클 ADAS HUD
**작성자:** 종현
**용도:** Colab에서 v6.5 폰 앱(HTML + YOLOv8n)을 패키징·배포하기 위한 노트북

> 본 노트북은 **Colab에서 폰 패키지를 만들어 다운로드**하기 위한 헬퍼입니다. 실제 ADAS 작동은 폰 브라우저에서 일어납니다.

## 백엔드 vs 폰
| | 백엔드 (v8b.3) | 폰 (v6.5) |
|---|---|---|
| 모델 | YOLO11m + SegFormer-B5 + YOLOPv2 + Flow | YOLOv8n + OpenCV.js |
| 환경 | Colab A100, 80ms/frame | 폰 브라우저, 28fps |
| 용도 | 오프라인 분석 (정지 바이크 필터 등) | 실시간 ADAS HUD |

---
## ✅ 사전 준비

Drive에 다음을 미리 업로드:

```
GUARDIAN/phone/
├── guardian_v6_5.html       (HTML 본체)
└── yolov8n_tfjs/
    ├── model.json
    ├── group1-shard1of4.bin
    ├── group1-shard2of4.bin
    ├── group1-shard3of4.bin
    └── group1-shard4of4.bin
```

## 1️⃣ 환경 설정 + Drive 마운트

In [ ]:
# ============================================================
# 환경 설정
# ============================================================
import os, shutil, zipfile
from pathlib import Path

# Drive 마운트
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

# 입력 (Drive)
SRC_HTML  = Path('/content/drive/MyDrive/GUARDIAN/phone/guardian_v6_5.html')
SRC_MODEL = Path('/content/drive/MyDrive/GUARDIAN/phone/yolov8n_tfjs')   # 폴더

# 출력
OUT_DIR  = Path('/content/drive/MyDrive/GUARDIAN/phone/dist')
OUT_DIR.mkdir(parents=True, exist_ok=True)
PACKAGE_NAME = 'guardian_v6_5'
ZIP_PATH = OUT_DIR / f'{PACKAGE_NAME}.zip'

print('✅ 경로 설정 완료')
print(f'   SRC_HTML : {SRC_HTML}  (exists: {SRC_HTML.exists()})')
print(f'   SRC_MODEL: {SRC_MODEL}  (exists: {SRC_MODEL.exists()})')
print(f'   OUT_DIR  : {OUT_DIR}')

## 2️⃣ 패키지 폴더 구성 + 압축

HTML과 모델 파일을 한 폴더로 모아서 zip으로 압축합니다.

In [ ]:
# ============================================================
# 패키지 빌드
# ============================================================
work_dir = Path('/tmp') / PACKAGE_NAME
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

# HTML 복사 (이름은 index.html로)
if not SRC_HTML.exists():
    raise FileNotFoundError(f'HTML 없음: {SRC_HTML}')
shutil.copy(SRC_HTML, work_dir / 'index.html')
print(f'✅ HTML  : {SRC_HTML.name} → index.html')

# 모델 폴더의 모든 파일 복사
if not SRC_MODEL.exists():
    raise FileNotFoundError(f'모델 폴더 없음: {SRC_MODEL}')
for fp in sorted(SRC_MODEL.iterdir()):
    if fp.is_file():
        shutil.copy(fp, work_dir / fp.name)
        print(f'✅ 모델  : {fp.name}')

# 압축
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(work_dir.iterdir()):
        if fp.is_file():
            zf.write(fp, arcname=f'{PACKAGE_NAME}/{fp.name}')

sz_mb = ZIP_PATH.stat().st_size / 1e6
print(f'\n📦 압축 완료: {ZIP_PATH.name} ({sz_mb:.1f} MB)')
print(f'   경로: {ZIP_PATH}')

## 3️⃣ 다운로드 (Colab → PC)

PC로 zip을 다운로드합니다. 이후 폰에 배포할 수 있습니다.

In [ ]:
# ============================================================
# Colab에서 직접 다운로드
# ============================================================
from google.colab import files
files.download(str(ZIP_PATH))
print('📥 다운로드 시작')

## 4️⃣ 폰 배포 가이드

### 방법 1 — Netlify Drop (추천, 5초만에 됨)
1. PC에서 위에서 다운로드한 zip 풀기
2. 폴더 통째로 https://app.netlify.com/drop 에 드래그
3. 자동 생성된 URL을 폰 브라우저에서 열기
4. 카메라 권한 허용 → ADAS 시작

### 방법 2 — 로컬 PC 서버 (같은 WiFi)
1. PC에서 zip 풀기
2. PC 터미널: `cd guardian_v6_5 && python3 -m http.server 8080`
3. 폰 같은 WiFi 연결
4. 폰 브라우저: `http://<PC IP>:8080`

> ⚠️  모바일 카메라는 HTTPS 필요. localhost만 HTTP에서 카메라 작동.
>    WiFi에서 작동하려면 ngrok 같은 HTTPS 터널 필요.

### 방법 3 — GitHub Pages
1. zip 풀어서 GitHub repo로 푸시
2. Settings → Pages → main 브랜치 활성화
3. `https://<유저>.github.io/<repo>` 폰에서 열기

---

## ✅ 폰에서 작동 확인 체크리스트
- [ ] 카메라 권한 허용
- [ ] 모델 로딩 완료 (12MB, 첫 로드시 5초)
- [ ] 객체 박스 표시 (CAR/PERSON 등)
- [ ] 거리 라벨 표시 (xxm)
- [ ] 속도계 작동 (GPS 권한)
- [ ] 차선 이탈 시 좌/우 빨간 막대
- [ ] 전방 충돌 시 ⚠ 깜박

## 🎬 HUD 구성 요소
- 좌상단: GUARDIAN | DTM 로고
- 우상단: REC 타이머
- 좌측: ENVIRONMENT 카드 (Mode, Risk Level)
- 우측: DETECTED OBJECTS 카드 (PERS, MOTO, CAR, TRUC, BUS 카운트)
- 중앙: 객체 박스 + 거리 표시 (CAR 87% 22m)
- 우측 위: 속도계 (km/h)
- 우측: FCW (Forward Collision Warning) 인디케이터
- 좌측 끝/우측 끝: LDW (Lane Departure Warning) 빨간 막대
- 좌하단: BPM (심박수, 시뮬레이션)